In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

# Image

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [ ]:
# TODO: Read image data, feed into Claude

with open("019_images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt
    }
])

response = chat(messages)
response


Message(id='msg_01UayaBcz7te8EJiAYh89S7o', container=None, content=[TextBlock(citations=None, text='# Satellite Image Analysis - Fire Risk Assessment\n\n## 1. Residence Identification\nThe primary residence is a single-story structure with a light-colored (gray/tan) roof located in the center of the image, appearing as an L-shaped or irregular rectangular building completely surrounded by dense forest vegetation.\n\n## 2. Tree Overhang Analysis\nMultiple trees with dense canopies directly overhang the residence roof, with overhanging vegetation covering approximately 50-75% of the visible roof surface, particularly concentrated on the northern and eastern portions of the structure.\n\n## 3. Fire Risk Assessment\nThe overhanging trees create multiple ember catch points across the roof surface and establish direct fuel continuity between the surrounding forest canopy and the structure, with branches appearing to touch or come very close to the roofline in several locations, creating sign

In [5]:
from IPython.display import Markdown, display

display(Markdown(text_from_message(response)))

# Satellite Image Analysis - Fire Risk Assessment

## 1. Residence Identification
The primary residence is a single-story structure with a light-colored (gray/tan) roof located in the center of the image, appearing as an L-shaped or irregular rectangular building completely surrounded by dense forest vegetation.

## 2. Tree Overhang Analysis
Multiple trees with dense canopies directly overhang the residence roof, with overhanging vegetation covering approximately 50-75% of the visible roof surface, particularly concentrated on the northern and eastern portions of the structure.

## 3. Fire Risk Assessment
The overhanging trees create multiple ember catch points across the roof surface and establish direct fuel continuity between the surrounding forest canopy and the structure, with branches appearing to touch or come very close to the roofline in several locations, creating significant wildfire vulnerability.

## 4. Defensible Space Identification
The property shows minimal defensible space, with a nearly continuous dense tree canopy extending over and immediately adjacent to the residence, creating clear fuel ladders from ground vegetation through tree canopies directly to the roof structure.

## 5. Fire Risk Rating
**Rating: 4 (Severe Risk)** - The property exhibits extensive tree overhang (50-75% of roof), dense vegetation in immediate contact with the structure, virtually no defensible space, continuous canopy fuel connections, and numerous ember catch points, creating extreme wildfire vulnerability with limited options for structure defense.

# PDF

In [ ]:
with open("019_earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    # Block for PDF
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "Summarize the document in one sentence"
    }
])

response = chat(messages)
response


Message(id='msg_019sAZwg8vgCHsQxgFx35u95', container=None, content=[TextBlock(citations=None, text='This Wikipedia article provides comprehensive information about Earth, describing it as the third planet from the Sun and the only known astronomical object to harbor life, covering its physical characteristics, formation about 4.5 billion years ago, atmosphere, oceans, and the development of life that has significantly altered its surface and climate.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9625, output_tokens=66, server_tool_use=None, service_tier='standard'))

In [6]:
from IPython.display import Markdown, display

display(Markdown(text_from_message(response)))

This Wikipedia article provides comprehensive information about Earth, describing it as the third planet from the Sun and the only known astronomical object to harbor life, covering its physical characteristics, formation about 4.5 billion years ago, atmosphere, oceans, and the development of life that has significantly altered its surface and climate.

In [ ]:
with open("019_earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    # Block for PDF
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans were formed?"
    }
])

response = chat(messages)
response


Message(id='msg_01MvvGyfiFY68VKzimgwaguJ', container=None, content=[TextBlock(citations=None, text="According to the document, Earth's atmosphere and oceans were formed by **volcanic activity and outgassing**.\n\nThe process occurred as follows:\n\n1. **Water vapor from volcanic activity and outgassing** condensed into the oceans\n\n2. This was **augmented by water and ice from asteroids, protoplanets, and comets**\n\n3. According to one model mentioned, **sufficient water to fill the oceans may have been on Earth since it formed**\n\n4. **Atmospheric greenhouse gases** kept the oceans from freezing when the newly forming Sun had only 70% of its current luminosity\n\n5. By 3.5 billion years ago (Ga), **Earth's magnetic field was established**, which helped prevent the atmosphere from being stripped away by the solar wind\n\nSo in summary: volcanic outgassing created the initial atmosphere and water vapor, which condensed to form oceans, with additional contributions from extraterrestri

In [8]:
from IPython.display import Markdown, display

display(Markdown(text_from_message(response)))

According to the document, Earth's atmosphere and oceans were formed by **volcanic activity and outgassing**.

The process occurred as follows:

1. **Water vapor from volcanic activity and outgassing** condensed into the oceans

2. This was **augmented by water and ice from asteroids, protoplanets, and comets**

3. According to one model mentioned, **sufficient water to fill the oceans may have been on Earth since it formed**

4. **Atmospheric greenhouse gases** kept the oceans from freezing when the newly forming Sun had only 70% of its current luminosity

5. By 3.5 billion years ago (Ga), **Earth's magnetic field was established**, which helped prevent the atmosphere from being stripped away by the solar wind

So in summary: volcanic outgassing created the initial atmosphere and water vapor, which condensed to form oceans, with additional contributions from extraterrestrial sources like asteroids and comets.

In [ ]:
# Citations with PDF - CitationPageLocation

messages = []

add_user_message(messages, [
    # Block for PDF
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }, 
        "title": "earth.pdf",
        "citations": { "enabled": True }
    },
    # Text Block
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans were formed?"
    }
])

response = chat(messages)
response


Message(id='msg_01RprGvqHXiS85K9aPFxRQLo', container=None, content=[TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationPageLocation(cited_text='[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n', document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-sonnet-4-5-20250929', ro